Assignment 6: Retrieval-Augmented Generation (RAG) - Machine Learning Knowledge
Assistant

Installing required libraries

In [1]:
!pip install -q pypdf langchain langchain-community langchain-google-genai google-generativeai faiss-cpu numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [4]:
import os
import re
import warnings
import numpy as np
from getpass import getpass

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_classic.schema import Document
from langchain_core.prompts import PromptTemplate

warnings.filterwarnings('ignore')

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ").strip()

PDF_PATH = "/content/intro-to-ml.pdf"   # <-- path to your ML book PDF

Enter your Google API key: ··········


Part 1 — Data Understanding & Preprocessing

In [5]:
reader    = PdfReader(PDF_PATH)
all_pages = [page.extract_text() or "" for page in reader.pages]

Exploring document structure

In [6]:
char_lengths = [len(p) for p in all_pages]

print(f"Total pages         : {len(all_pages)}")
print(f"Avg chars per page  : {int(np.mean(char_lengths))}")
print(f"Min / Max chars     : {min(char_lengths)} / {max(char_lengths)}")
print(f"Near-empty pages    : {sum(1 for c in char_lengths if c < 50)}")
print("\n--- Sample text (page 5) ---")
print(all_pages[4][:400])

Total pages         : 392
Avg chars per page  : 1765
Min / Max chars     : 0 / 4604
Near-empty pages    : 5

--- Sample text (page 5) ---
Table of Contents
Preface. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  vii
1. Introduction. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  1
Why Machine Learning?                                                             


Text quality check

In [7]:
broken_words = sum(len(re.findall(r'\w+-\n\w+', p)) for p in all_pages)
extra_spaces = sum(len(re.findall(r' {3,}', p))     for p in all_pages)
number_only  = sum(1 for p in all_pages if re.fullmatch(r'\s*\d+\s*', p))

print(f"Hyphenated line-breaks : {broken_words}")
print(f"Excessive spaces       : {extra_spaces}")
print(f"Page-number-only pages : {number_only}")

Hyphenated line-breaks : 55
Excessive spaces       : 1572
Page-number-only pages : 0


Cleaning the text

In [8]:
def clean_page(text):
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)                    # fix hyphenated words
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)   # remove lone page numbers
    text = re.sub(r' {2,}', ' ', text)                                  # collapse extra spaces
    text = re.sub(r'\n{3,}', '\n\n', text)                             # collapse blank lines
    return text.strip()

cleaned_pages = [clean_page(p) for p in all_pages]

Split text into chunks
- chunk_size = 800 — enough text for meaningful context per chunk  
- chunk_overlap = 120 — avoids losing context at chunk boundaries

In [9]:
full_text = "\n\n".join(p for p in cleaned_pages if len(p) > 50)

splitter  = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)
documents = [Document(page_content=chunk) for chunk in splitter.split_text(full_text)]

print(f"Total chunks : {len(documents)}")
print(f"Avg chars    : {int(np.mean([len(d.page_content) for d in documents]))}")
print("\n--- Sample chunk ---")
print(documents[10].page_content)

Total chunks : 1110
Avg chars    : 660

--- Sample chunk ---
Grid Search with Cross-Validation 263
Evaluation Metrics and Scoring 275
Keep the End Goal in Mind 275
Metrics for Binary Classification 276
Metrics for Multiclass Classification 296
Regression Metrics 299
Using Evaluation Metrics in Model Selection 300
Summary and Outlook 302
6. Algorithm Chains and Pipelines. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 305
Parameter Selection with Preprocessing 306
Building Pipelines 308
Using Pipelines in Grid Searches 309
The General Pipeline Interface 312
Convenient Pipeline Creation with make_pipeline 313
Accessing Step Attributes 314
Accessing Attributes in a Grid-Searched Pipeline 315
Grid-Searching Preprocessing Steps and Model Parameters 317
Grid-Searching Which Model To Use 319
Summary and Outlook 320


Part 2 — Embedding & Vector Database

Embedding model:  `gemini-embedding-001` —
Vector store: FAISS — stores all vectors and retrieves the closest matches for any query

In [10]:
embedding_model = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001')
vector_store    = FAISS.from_documents(documents, embedding_model)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 9.386326109s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', '

Sample Retreival

In [ ]:
test_query  = "How does a neural network learn from data?"
top_results = vector_store.similarity_search_with_score(test_query, k=3)

print(f"Query: \"{test_query}\"\n")
for rank, (doc, score) in enumerate(top_results, 1):
    print(f"Rank {rank} | Score: {score:.4f}")
    print(doc.page_content[:250])
    print("-" * 55)

Part 3 — Retrieval Pipeline

In [ ]:
def fetch_chunks(question, k=5):
    return vector_store.similarity_search_with_score(question, k=k)

Experimenting with k = 3,4,5

In [11]:
exp_query = "What is regularisation and why is it needed?"

for k in [3, 5, 7]:
    hits = fetch_chunks(exp_query, k=k)
    print(f"\n=== k={k} ===")
    for i, (doc, score) in enumerate(hits, 1):
        print(f"  [{i}] Score: {score:.4f} | {doc.page_content[:150]}...")

NameError: name 'fetch_chunks' is not defined

Part 4 — Answer Generation (RAG)


In [ ]:
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""\
You are a helpful tutor specialising in Machine Learning and Data Science.
Answer the student's question using ONLY the reference material provided below.

Rules:
- Do not use any knowledge outside the reference material.
- If the material does not contain the answer, say: "The provided material does not cover this topic."
- Use bullet points or numbered steps when explaining a process.

Reference Material:
{context}

Student Question: {question}

Answer:"""
)

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1)



In [ ]:
def answer_question(question, k=5):
    hits     = fetch_chunks(question, k=k)
    context  = "\n\n".join(doc.page_content for doc, _ in hits)
    prompt   = RAG_PROMPT.format(context=context, question=question)
    response = llm.invoke(prompt)
    return response.content if hasattr(response, 'content') else response

Sample Queries

In [ ]:
questions = [
    "What is the purpose of an activation function in a neural network?",
    "How does the random forest algorithm reduce overfitting?",
    "What is the difference between supervised and unsupervised learning?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {answer_question(q)}")

Hallucination Test

In [12]:
out_of_scope = "What is the GDP of Germany in 2023?"
print(f"Q: {out_of_scope}")
print(f"A: {answer_question(out_of_scope)}")

Q: What is the GDP of Germany in 2023?


NameError: name 'answer_question' is not defined

Part 5 - End to End RAG Pipeline

In [ ]:
def rag_qa(question, k=5, show_sources=False):
    hits     = fetch_chunks(question, k=k)
    context  = "\n\n".join(doc.page_content for doc, _ in hits)
    prompt   = RAG_PROMPT.format(context=context, question=question)
    response = llm.invoke(prompt)
    answer   = response.content if hasattr(response, 'content') else response

    print(f"\nQ: {question}")
    if show_sources:
        print("\n[Sources]")
        for i, (doc, score) in enumerate(hits, 1):
            print(f"  ({i}) Score {score:.4f} | {doc.page_content[:120]}...")
    print(f"\nA: {answer}")

In [ ]:
demo_questions = [
    "How does gradient descent update model weights during training?",
    "What is the role of the loss function in machine learning?",
    "Explain how principal component analysis reduces the number of features.",
    "What happens when a model has high bias and low variance?",
    "How does the k-means algorithm assign data points to clusters?",
]

for q in demo_questions:
    rag_qa(q, k=5)

In [ ]:
# With source passages visible
rag_qa("What is the vanishing gradient problem and how is it addressed?", k=5, show_sources=True)

In [ ]:
while True:
    user_q = input("Ask a question (or type 'exit'): ").strip()
    if not user_q:
        continue
    if user_q.lower() in ("exit", "quit"):
        break
    rag_qa(user_q, k=5)